<a href="https://colab.research.google.com/github/tsukki8/ds2002-fa26/blob/main/notebooks/04-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")

print(f"400 orders generated ${total_revenue:,.2f} "
      f"from {total_units} total units.")

Total revenue: $8,520.00
Total units: 783
400 orders generated $8,520.00 from 783 total units.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (
    df.groupby('category')
      .agg(revenue=('revenue', 'sum'))
      .sort_values('revenue', ascending=False)
)

by_category['share_pct'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
)

by_category['share_pct'] = by_category['share_pct'].round(1)

print(by_category)

print(
    f"\nHighest revenue at "
    f"${by_category.loc['Food', 'revenue']:,.2f}, "
    f"{by_category.loc['Food', 'share_pct']:.1f}% "
    f"of total revenue."
)

          revenue  share_pct
category                    
Food       4293.0       50.4
Merch      1771.5       20.8
Drink      1554.0       18.2
RainGear    901.5       10.6

Highest revenue at $4,293.00, 50.4% of total revenue.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = (
    df.groupby('vendor_id')
      .agg(
          avg_order_revenue=('revenue', 'mean'),
          order_count=('revenue', 'count')
      )
      .sort_values('avg_order_revenue', ascending=False)
)

by_vendor['avg_order_revenue'] = by_vendor['avg_order_revenue'].round(2)

print(by_vendor)

top_vendor = by_vendor.index[0]

print(
    f"\n{top_vendor} has the highest average order revenue at "
    f"${by_vendor.loc[top_vendor, 'avg_order_revenue']:.2f} "
    f"across {by_vendor.loc[top_vendor, 'order_count']} orders."
)

           avg_order_revenue  order_count
vendor_id                                
V-01                   22.60           94
V-18                   21.75          108
V-05                   20.58           93
V-10                   20.31          105

V-01 has the highest average order revenue at $22.60 across 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = merch_revenue / df['revenue'].sum() * 100

print(f"Merch revenue share: {merch_share:.1f}%")

print(
    f"Merch accounted for {merch_share:.1f}% of total revenue, "
    f"or ${merch_revenue:,.2f} out of ${df['revenue'].sum():,.2f}."
)

Merch revenue share: 20.8%
Merch accounted for 20.8% of total revenue, or $1,771.50 out of $8,520.00.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one',
    indicator=True
)

print("Rows before merge:", len(df))
print("Rows after merge:", len(joined))

print("Revenue before merge:", df['revenue'].sum())
print("Revenue after merge:", joined['revenue'].sum())

unmatched = joined.loc[joined['_merge'] == 'left_only', 'vendor_id'].unique()

print("Unmatched vendor(s):", unmatched)

print(
    f"\nThe merge preserved all {len(joined)} rows and "
    f"the total revenue remained ${joined['revenue'].sum():,.2f}."
)

Rows before merge: 400
Rows after merge: 400
Revenue before merge: 8520.0
Revenue after merge: 8520.0
Unmatched vendor(s): ['V-18']

The merge preserved all 400 rows and the total revenue remained $8,520.00.


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)

print(pivot)

category         Drink    Food   Merch  RainGear   Total
vendor_name                                             
Cav Merch North  502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers     171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos    298.5   882.0   489.0     244.5  1914.0
Total            972.0  3274.5  1263.0     661.5  6171.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell these vendors to sell mainly food because in the revenue by category query, the highest category revenus is food with 50.4% share of the total revenue. Next category is merch, which is 20.8% share of the total. I would recommend a variety of products to cover the smaller percentages of revenue from drinks and prepare raingear seasonally as well.

b) The least trustworthy answer is the join in the vendor names because of the unmatched vendor V-18. This specific unmatched vendor will lead to skew in the data because although the vendor was kept in the data, the V-18 was not associated with a vendor name.